In [3]:
from d3rlpy.algos import BCQ, BCQConfig
from d3rlpy.models.encoders import VectorEncoderFactory

vae_encoder = VectorEncoderFactory([750, 750])
rl_encoder = VectorEncoderFactory([400, 300])

In [10]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

from d3rlpy.dataset import ReplayBuffer, FIFOBuffer
import argparse
import d3rlpy
import gymnasium as gym


dataset = "hopper-medium-expert-v2"
if "halfcheetah" in dataset:
    env_name = "HalfCheetah-v4"
elif "hopper" in dataset:
    env_name = "Hopper-v4"
elif "walker" in dataset:
    env_name = "Walker2d-v4"

#env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v5"
print(env_name)
on_server = False
if on_server:
    prefix = "/gpfs/data/fs72297/jklotz/programming/cloned_repos/master_thesis/reproducing_decision_transformer/gymnasium/data/"
else:
    prefix = "/home/julian/programming/cloned_repos/master_thesis/reproducing_decision_transformer/gymnasium/data/"
pkl_path = f"{prefix}{dataset}.pkl"

def convert_raw_episode(raw_ep):
    # Convert raw observations to a NumPy array and then to a list of individual observations.
    observations = np.array(raw_ep["observations"])

    # Ensure actions and rewards are NumPy arrays.
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    # For rewards, ensure they have an extra dimension (T, 1)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # Use the last element of "terminals" as the terminated flag.
    terminals = raw_ep["terminals"]
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)
print(f"Loaded {len(episodes)} episodes.")


buffer_impl = FIFOBuffer(limit=10000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)



args = argparse.Namespace()
args.dataset = dataset
args.seed = 1
args.gpu = "cuda:0" if on_server else "cpu"
args.compile = False


env = gym.make(env_name)

#dataset, env = d3rlpy.datasets.get_dataset(args.dataset)

# fix seed
d3rlpy.seed(args.seed)
d3rlpy.envs.seed_env(env, args.seed)

if "halfcheetah" in args.dataset:
    target_return = 6000
elif "hopper" in args.dataset:
    target_return = 3800
elif "walker" in args.dataset:
    target_return = 5000
else:
    raise ValueError("unsupported dataset")

Hopper-v4
Loaded 3213 episodes.
2025-07-18 12:44.45 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-07-18 12:44.45 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-07-18 12:44.45 [info     ] Action size has been automatically determined. action_size=3


/home/julian/miniconda3/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/gymnasium/envs/registration.py:517: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


In [ ]:
bcq = BCQConfig(
        actor_encoder_factory=rl_encoder,
        actor_learning_rate=1e-3,
        critic_encoder_factory=rl_encoder,
        critic_learning_rate=1e-3,
        imitator_encoder_factory=vae_encoder,
        imitator_learning_rate=1e-3,
        batch_size=100,
        lam=0.75,
        action_flexibility=0.05,
        n_action_samples=100,
        compile_graph=args.compile,
    ).create(args.gpu)

bcq.fit(
    replay_buffer,
    n_steps=500,#500000
    n_steps_per_epoch=10,#1000
    save_interval=10,
    evaluators={"environment": d3rlpy.metrics.EnvironmentEvaluator(env)},
    experiment_name=f"BCQ_{args.dataset}_{args.seed}",
    )

2025-07-18 12:44.48 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-07-18 12:44.48 [debug    ] Building models...            
2025-07-18 12:44.50 [debug    ] Models have been built.       
2025-07-18 12:44.50 [info     ] Directory is created at d3rlpy_logs/BCQ_hopper-medium-expert-v2_1_20250718124450
2025-07-18 12:44.50 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'bcq', 'params': {'batch_size': 100, 'gamma': 0.99, 'observation_scaler': {'type': 'none', 'params': {}}, 'action_scaler': {'type': 'none', 'params': {}}, 'reward_scaler': {'type': 'none', 'params': {}}, 'compile_graph': False, 'actor_learning_rate': 0.001, 'critic_learn

Epoch 1/50: 100%|██████████| 10/10 [00:02<00:00,  3.71it/s, vae_loss=0.51, critic_loss=23.2, actor_loss=-0.807]


2025-07-18 12:44.55 [info     ] BCQ_hopper-medium-expert-v2_1_20250718124450: epoch=1 step=10 epoch=1 metrics={'time_sample_batch': 0.011659884452819824, 'time_algorithm_update': 0.2556295394897461, 'vae_loss': 0.5532271027565002, 'critic_loss': 6.711675715446472, 'actor_loss': -3.1055132269859316, 'time_step': 0.2675044298171997, 'environment': 48.11708956009783} step=10


Epoch 2/50: 100%|██████████| 10/10 [00:02<00:00,  4.19it/s, vae_loss=0.333, critic_loss=1.98, actor_loss=-3.14]


2025-07-18 12:45.00 [info     ] BCQ_hopper-medium-expert-v2_1_20250718124450: epoch=2 step=20 epoch=2 metrics={'time_sample_batch': 0.010904455184936523, 'time_algorithm_update': 0.22577967643737792, 'vae_loss': 0.3030183523893356, 'critic_loss': 1.4901237487792969, 'actor_loss': -3.0597151279449464, 'time_step': 0.23686592578887938, 'environment': 160.25783807881805} step=20


...

Epoch 5/50: 100%|██████████| 10/10 [00:02<00:00,  3.85it/s, vae_loss=0.244, critic_loss=0.217, actor_loss=-4.16]


2025-07-18 12:45.15 [info     ] BCQ_hopper-medium-expert-v2_1_20250718124450: epoch=5 step=50 epoch=5 metrics={'time_sample_batch': 0.011595368385314941, 'time_algorithm_update': 0.24656109809875487, 'vae_loss': 0.23009719848632812, 'critic_loss': 0.17059883028268813, 'actor_loss': -4.151708364486694, 'time_step': 0.2583733320236206, 'environment': 118.66688875659251} step=50


Epoch 6/50: 100%|██████████| 10/10 [00:02<00:00,  3.66it/s, vae_loss=0.258, critic_loss=0.0905, actor_loss=-4.44]


2025-07-18 12:45.21 [info     ] BCQ_hopper-medium-expert-v2_1_20250718124450: epoch=6 step=60 epoch=6 metrics={'time_sample_batch': 0.013766860961914063, 'time_algorithm_update': 0.25771231651306153, 'vae_loss': 0.23874864131212234, 'critic_loss': 0.09080754593014717, 'actor_loss': -4.376881837844849, 'time_step': 0.27168965339660645, 'environment': 143.1662802998354} step=60


Epoch 7/50:  40%|████      | 4/10 [00:01<00:01,  3.84it/s, vae_loss=0.217, critic_loss=0.154, actor_loss=-4.45]